In [ ]:
# Cell 1 — Mount Drive and find the project folder
from google.colab import drive
from pathlib import Path
import os
import sys

drive.mount("/content/drive")


def find_project_dir() -> Path:
    marker_files = [
        "colab_utils/requirements_colab.txt",
        "colab_utils/colab_launcher.py",
        "app/chatbot.py",
    ]

    search_roots = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    candidates = []

    for root in search_roots:
        if not root.exists():
            continue

        for requirements_file in root.rglob("requirements_colab.txt"):
            candidate = requirements_file.parents[1]

            if all((candidate / marker).exists() for marker in marker_files):
                candidates.append(candidate)

    candidates = sorted(set(candidates), key=lambda path: str(path).lower())

    if not candidates:
        raise FileNotFoundError(
            "Project folder not found. If the folder was shared with you, "
            "open Google Drive, go to 'Shared with me', right-click the project folder, "
            "choose 'Add shortcut to Drive', then run this cell again."
        )

    if len(candidates) == 1:
        return candidates[0]

    print("Multiple project folders found:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}. {candidate}")

    choice = input("Choose the project folder number, or press Enter to use the first one: ").strip()
    if not choice:
        return candidates[0]

    return candidates[int(choice) - 1]


PROJECT_DIR = find_project_dir()
COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)

In [ ]:
# Cell 2 — Install/update dependencies
# Run this after a fresh runtime. If the environment is already ready, you can skip it.
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH" 

In [ ]:
# Cell 3 — Environment and debug configuration
import os
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not found. Public models can still be loaded if access is not restricted.")

APP_DEBUG = True
os.environ["APP_DEBUG"] = str(APP_DEBUG).lower()

logging.basicConfig(
    level=logging.DEBUG if APP_DEBUG else logging.WARNING,
    format="%(levelname)s:%(message)s",
    force=True,
)

print(f"APP_DEBUG={os.environ['APP_DEBUG']}")

In [ ]:
# Cell 4 — Reload project modules from Google Drive
# Run this cell after editing project files. It does not reload the LLM object.
import os
import sys
import importlib
from pathlib import Path

if "PROJECT_DIR" not in globals():
    raise RuntimeError("PROJECT_DIR is not defined. Run Cell 1 first.")

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

Path("app/__init__.py").touch()
importlib.invalidate_caches()

MODULES_TO_REFRESH = [
    "settings",
    "utils.settings",
    "prompts.router_prompt",
    "prompts.nlu_prompt",
    "prompts.dm_prompt",
    "prompts.nlg_prompt",
    "llm.generation",
    "llm.config",
    "llm.loader",
    "components.router",
    "components.NLU",
    "components.DM",
    "components.NLG",
    "state.dialogue_state_tracker",
    "state.history",
    "state.task_queue",
    "database.db_controller",
    "database.mock_database",
    "app.chatbot",
]

for module_name in MODULES_TO_REFRESH:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

from app.chatbot import Chatbot
from components.router import Router
from components.NLU import NLU
from components.DM import DM
from components.NLG import NLG
from state.dialogue_state_tracker import StateTracker
from state.history import History
from database.db_controller import DBController
from state.task_queue import TaskQueue
from llm.loader import load_llm

print("Project modules loaded from:", PROJECT_DIR)
print("If you edited components, prompts, DB, state, or app.chatbot, now rerun the bot initialization cell.")
print("If you edited llm.loader, llm.config, or llm.generation, rerun the LLM loading cell too.")

In [ ]:
# Cell 5 — Load the LLM once
# Run this only when you need to download/reload the model.
MODEL_NAME = "qwen3"

llm = load_llm(MODEL_NAME)

print(f"LLM loaded once: {MODEL_NAME}")

In [ ]:
# Cell 6 — Initialize the chatbot with the already-loaded LLM
# This uses the current Chatbot methods from app.chatbot, but avoids calling Chatbot.__init__()
# because __init__ would load the LLM again.

def build_bot_from_loaded_llm(llm):
    bot = Chatbot.__new__(Chatbot)

    bot.llm = llm
    bot.router = Router(llm)
    bot.NLU = NLU(llm)
    bot.DM = DM(llm)
    bot.NLG = NLG(llm)

    bot.dst = StateTracker()
    bot.db_controller = DBController(bot.dst)
    bot.history = History()
    bot.task_queue = TaskQueue()

    bot.DONE_STATUSES = ["INFORM", "CONFIRMED", "ABORTED"]

    return bot

bot = build_bot_from_loaded_llm(llm)

print("Bot initialized with the latest loaded project modules.")
print("You can now test with bot.reply(...) or bot.chat_loop().")

In [ ]:
# Cell 8 — Interactive console loop
# Type exit, quit, or stop to end the loop.
bot.chat_loop()

In [ ]:
# Cell 9 — Inspect conversation history
try:
    bot.history.print_full_conversation()
except Exception as error:
    print(f"Could not print history: {error}")

In [ ]:
# Cell 11 — Reset conversation state without reloading the model
bot.reset_state()
print("Conversation state reset. The LLM is still loaded.")